In [25]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [ ]:
sys.path.append(str(Path.cwd().parent))

ROOT    = Path(os.getcwd()).resolve().parents[0]
DATA    = ROOT / "data"
WEIGHTS = ROOT / "weights"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", train_df.shape)

Train set: (1708, 12)
Test set: (1708, 12)


## Load BERT Models

In [27]:
import torch
from dataclasses import dataclass, field
from transformers import (
    BertForQuestionAnswering, 
    AutoTokenizer,
)
from sentence_transformers import SentenceTransformer

In [28]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [29]:
bert_model = BertForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
bert_model.to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2').to(device)

# Count BERT's total number of trainable parameters by summing the size of every weight tensor.
bert_num_params = sum(p.numel() for p in bert_model.parameters())
# Print the count, so "small model" isn't just a label in a markdown cell -- see the real number.
print(f"BERT ({bert_model.config.name_or_path}) parameters: {bert_num_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5622.09it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7706.40it/s]


BERT (deepset/bert-base-cased-squad2) parameters: 107,721,218


## Probe Training Configuration

In [30]:
@torch.no_grad()
def compute_baselines(model, loader, device):
    model.eval()

    all_questions = []
    all_human_style = []
    all_agent_style = []
    all_human_content = []
    all_agent_content = []
    all_human_context = []
    all_agent_context = []

    for batch in loader:
        question = batch["question"].to(device)
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        # Encode the paired human and AI responses
        style_h, content_h, context_h = model.encode(human)
        style_a, content_a, context_a = model.encode(agent)

        all_questions.append(question.cpu())
        all_human_style.append(style_h.cpu())
        all_agent_style.append(style_a.cpu())
        all_human_content.append(content_h.cpu())
        all_agent_content.append(content_a.cpu())
        all_human_context.append(context_h.cpu())
        all_agent_context.append(context_a.cpu())

    # Concatenate all examples across batches
    human_style = torch.cat(all_human_style, dim=0)
    agent_style = torch.cat(all_agent_style, dim=0)
    human_content = torch.cat(all_human_content, dim=0)
    agent_content = torch.cat(all_agent_content, dim=0)
    human_context = torch.cat(all_human_context, dim=0)
    agent_context = torch.cat(all_agent_context, dim=0)

    # Pool human and agent targets for each representation.
    # Each response receives equal weight.
    all_content = torch.cat([human_content, agent_content], dim=0)
    all_context = torch.cat([human_context, agent_context], dim=0)

    # Calculate the mean target vectors
    baselines = {
        "content": all_content.mean(dim=0),
        "context": all_context.mean(dim=0),
    }

    return baselines

In [31]:
from dataclasses import dataclass

@dataclass
class Config:
    epochs: int = 50

    # Data shape (fixed by SBERT)
    question_dim: int = sentence_model.get_sentence_embedding_dimension()
    bert_dim: int     = sentence_model.get_sentence_embedding_dimension()

    # Code sizes
    style_dim: int    = sentence_model.get_sentence_embedding_dimension()
    content_dim: int  = sentence_model.get_sentence_embedding_dimension()
    context_dim: int  = sentence_model.get_sentence_embedding_dimension()

    # Hidden sizes
    style_hidden: int   = 16
    content_hidden: int = 128
    context_hidden: int = 128

    # Probe outputs: filled in from the code sizes below
    style_out: int    = 1
    content_out: int  = None
    context_out: int  = None

    disc_dim: int     = sentence_model.get_sentence_embedding_dimension()
    gene_dim: int     = sentence_model.get_sentence_embedding_dimension()

    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

    w_align: float      = 1.0
    w_leak: float       = 1.0
    lr_probe: float     = 1e-3
    w_recon: float      = 1.0
    w_style: float      = 1.0
    w_content: float    = 1.0
    w_context: float    = 1.0
    w_retrieval: float  = 1.0

    def __post_init__(self):
        if self.content_out is None:
            self.content_out = self.content_dim
        if self.context_out is None:
            self.context_out = self.context_dim

cfg = Config(epochs=50)

/tmp/ipykernel_15448/2819680373.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  question_dim: int = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_15448/2819680373.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bert_dim: int     = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_15448/2819680373.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  style_dim: int    = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_15448/2819680373.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  content_dim: int  = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_15448/2819680373.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_e

In [32]:
alt_state = torch.load(WEIGHTS / "alt_state.pt", weights_only=False, map_location="cpu")
mix_state = torch.load(WEIGHTS / "mix_state.pt", weights_only=False, map_location="cpu")

alt_config = alt_state["config"]
mix_config = mix_state["config"]

alt_model = alt_state["model"]
mix_model = mix_state["model"]

alt_model.to(device)
mix_model.to(device)

SBERTDisentangler(
  (style_encoder): StyleEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=16, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=16, out_features=384, bias=True)
    )
  )
  (content_encoder): ContentEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=384, bias=True)
    )
  )
  (context_encoder): ContextEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=384, bias=True)
    )
  )
  (generator): SBERTGenerator(
    (mlp): Sequential(
      (0): Linear(in_features=1152, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (discriminator): SBERTDiscriminator(
    (mlp): Sequen

In [ ]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

## Evaluate Probes

In [34]:
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import r2_score

In [35]:
probe_specs = {
    "style_content":   ("style", "content"),
    "style_context":   ("style", "context"),
    "content_style":   ("content", "style"),
    "content_context": ("content", "context"),
    "context_style":   ("context", "style"),
    "context_content": ("context", "content"),
}

In [36]:
# Original SBERT response embeddings — no model.encode()
X_train = np.concatenate([np.stack(train_df["hr_bert"].to_numpy()), np.stack(train_df["as_bert"].to_numpy()),], axis=0)
X_test  = np.concatenate([np.stack(test_df["hr_bert"].to_numpy()),np.stack(test_df["as_bert"].to_numpy()),], axis=0)

# Source labels: 0 = human, 1 = AI
y_train = np.concatenate([np.zeros(len(train_df), dtype=int),np.ones(len(train_df), dtype=int),])
y_test  = np.concatenate([np.zeros(len(test_df), dtype=int),np.ones(len(test_df), dtype=int),])

# Linear classification probe
probe = make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000))

probe.fit(X_train, y_train)
predictions = probe.predict(X_test)

print(
    f"SBERT human/AI accuracy: "
    f"{accuracy_score(y_test, predictions):.4f}"
)
print("Chance accuracy: 0.5000")


SBERT human/AI accuracy: 0.8447
Chance accuracy: 0.5000


In [37]:

def evaluate_sbert_task_probe(
    train_embeddings,
    test_embeddings,
    y_train,
    y_test,
    task="classification",
):
    X_train = np.asarray(train_embeddings, dtype=np.float32)
    X_test = np.asarray(test_embeddings, dtype=np.float32)
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)

    if task == "classification":
        probe = make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000))
        probe.fit(X_train, y_train)
        predictions = probe.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        print(f"SBERT task accuracy: {accuracy:.4f}")
        return probe, {"accuracy": accuracy}

    elif task == "regression":
        probe = make_pipeline(StandardScaler(),Ridge(alpha=1.0))
        probe.fit(X_train, y_train)
        predictions = probe.predict(X_test)

        mse = mean_squared_error(y_test, predictions)
        print(f"SBERT task MSE: {mse:.6f}")
        return probe, {"mse": mse}

    else:
        raise ValueError(
            "task must be 'classification' or 'regression'"
        )


## Results
| Metric | Result | Interpretation |
|-|-|-|
| Human/AI Classification Accuracy | 0.7816 | Original SBERT embeddings contain substantial linearly accessible source information. |
| Regression cosine similarity | 1.0000 | Trivial self-reconstruction; not informative. |
| Regression MSE | 0.0000 | Trivial self-reconstruction; not informative. |
| Mean-vector baseline cosine | 0.4442 | Similarity to the average embedding, not a disentanglement metric. |

In [38]:
import copy

def reinit(model):
    m = copy.deepcopy(model)
    for enc in (m.style_encoder, m.content_encoder, m.context_encoder):
        for layer in enc.modules():
            if hasattr(layer, "reset_parameters"):
                layer.reset_parameters()
    return m

model_configs = [(reinit(mix_model), "random")]   # then run the existing probe loop unchanged

In [39]:
def retrieval(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    s, t = a @ b.T, np.arange(len(a))
    return ((s.argmax(1) == t).mean() + (s.argmax(0) == t).mean()) / 2

H, A = np.stack(test_df["hr_bert"]), np.stack(test_df["as_bert"])
print("raw human→agent retrieval:", retrieval(H, A))        # compare to content retrieval 0.51-0.53

# raw response → question (what any linear function of the response could recover)
Htr, Qtr = np.stack(train_df["hr_bert"]), np.stack(train_df["question_bert"])
Qte = np.stack(test_df["question_bert"])
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(
    np.r_[Htr, np.stack(train_df["as_bert"])], np.r_[Qtr, Qtr])
print("raw→question retrieval:", retrieval(ridge.predict(H), Qte))  # compare to context retrieval 0.27

raw human→agent retrieval: 0.6210526315789474
raw→question retrieval: 0.13157894736842105


In [40]:

results = {}

# Original SBERT embeddings, without the disentanglement model.
X_train = {
    "style": np.concatenate([
        np.stack(train_df["hr_bert"].to_numpy()),
        np.stack(train_df["as_bert"].to_numpy()),
    ]),
    "content": np.concatenate([
        np.stack(train_df["hr_bert"].to_numpy()),
        np.stack(train_df["as_bert"].to_numpy()),
    ]),
    "context": np.concatenate([
        np.stack(train_df["hr_bert"].to_numpy()),
        np.stack(train_df["as_bert"].to_numpy()),
    ]),
}

X_test = {
    "style": np.concatenate([
        np.stack(test_df["hr_bert"].to_numpy()),
        np.stack(test_df["as_bert"].to_numpy()),
    ]),
    "content": np.concatenate([
        np.stack(test_df["hr_bert"].to_numpy()),
        np.stack(test_df["as_bert"].to_numpy()),
    ]),
    "context": np.concatenate([
        np.stack(test_df["hr_bert"].to_numpy()),
        np.stack(test_df["as_bert"].to_numpy()),
    ]),
}

# 0 = human, 1 = AI
y_train = np.concatenate([np.zeros(len(train_df), dtype=int),np.ones(len(train_df), dtype=int),])
y_test = np.concatenate([np.zeros(len(test_df), dtype=int),np.ones(len(test_df), dtype=int),])

# Fit the mean-vector baselines on training data only.
baselines = {target: X_train[target].mean(axis=0) for target in ("content", "context")}

results["sbert"] = {}

for probe_name, (source, target) in probe_specs.items():
    Xtr = X_train[source]
    Xte = X_test[source]

    if target == "style":
        probe = make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000))

        probe.fit(Xtr, y_train)
        predictions = probe.predict(Xte)
        accuracy = accuracy_score(y_test, predictions)

        results["sbert"][probe_name] = {
            "accuracy": accuracy
        }

        print(
            f"{probe_name:20s} | "
            f"Accuracy: {accuracy:.4f} | Chance: 0.5000"
        )

    else:
        # Predict the original SBERT response embedding.
        Y_train = X_train[target]
        Y_test = X_test[target]

        probe = make_pipeline(StandardScaler(),Ridge(alpha=1.0))

        probe.fit(Xtr, Y_train)
        Y_pred = probe.predict(Xte)

        cosine = np.diag(cosine_similarity(Y_pred, Y_test)).mean()
        mse = mean_squared_error(Y_test, Y_pred)
        r2 = r2_score(Y_test, Y_pred)

        baseline_pred = np.broadcast_to(baselines[target], Y_test.shape)
        baseline_cosine = np.diag(cosine_similarity(baseline_pred, Y_test)).mean()
        baseline_mse = mean_squared_error(Y_test, baseline_pred)
        baseline_r2 = r2_score(Y_test, baseline_pred)

        results["sbert"][probe_name] = {
            "cosine": cosine,
            "mse": mse,
            "r2": r2,
            "baseline_cosine": baseline_cosine,
            "baseline_mse": baseline_mse,
            "baseline_r2": baseline_r2,
            "cosine_gain": cosine - baseline_cosine,
            "mse_reduction": baseline_mse - mse,
            "r2_gain": r2 - baseline_r2,
        }

        print(
            f"{probe_name:16s} | "
            f"Probe: Cos={cosine:.4f}, "
            f"MSE={mse:.6f}, R²={r2:.4f} | "
            f"Baseline: Cos={baseline_cosine:.4f}, "
            f"MSE={baseline_mse:.6f}, R²={baseline_r2:.4f} | "
            f"ΔCos={cosine - baseline_cosine:+.4f}, "
            f"ΔMSE={baseline_mse - mse:+.6f}, "
            f"ΔR²={r2 - baseline_r2:+.4f}"
        )


style_content    | Probe: Cos=1.0000, MSE=0.000000, R²=1.0000 | Baseline: Cos=0.2673, MSE=0.083243, R²=-0.0040 | ΔCos=+0.7327, ΔMSE=+0.083243, ΔR²=+1.0040
style_context    | Probe: Cos=1.0000, MSE=0.000000, R²=1.0000 | Baseline: Cos=0.2673, MSE=0.083243, R²=-0.0040 | ΔCos=+0.7327, ΔMSE=+0.083243, ΔR²=+1.0040
content_style        | Accuracy: 0.8447 | Chance: 0.5000
content_context  | Probe: Cos=1.0000, MSE=0.000000, R²=1.0000 | Baseline: Cos=0.2673, MSE=0.083243, R²=-0.0040 | ΔCos=+0.7327, ΔMSE=+0.083243, ΔR²=+1.0040
context_style        | Accuracy: 0.8447 | Chance: 0.5000
context_content  | Probe: Cos=1.0000, MSE=0.000000, R²=1.0000 | Baseline: Cos=0.2673, MSE=0.083243, R²=-0.0040 | ΔCos=+0.7327, ΔMSE=+0.083243, ΔR²=+1.0040
